<a href="https://colab.research.google.com/github/ipez02/cs166/blob/main/Copy_of_dqn_pong_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DQN Pong Project (Colab-ready)

This notebook implements a Baseline DQN for Atari Pong and includes toggles for Double DQN, Dueling DQN, Prioritized Replay, and N-step returns. Edit the **User settings** cell and run in Google Colab.

In [ ]:
# Install dependencies (run in Colab)
# If you are running locally, adapt installations to your environment.

# First, uninstall conflicting versions to ensure a clean install
!pip uninstall -y numpy opencv-python-headless

# Install numpy 1.x for compatibility with older opencv-python-headless versions
!pip install --quiet numpy==1.26.4

# Then install other dependencies, ensuring opencv-python-headless is compatible with numpy 1.x
!pip install --quiet gymnasium[atari,accept-rom-license] torch torchvision matplotlib imageio opencv-python-headless==4.7.0.72

# Install autorom separately.
!pip install autorom

# autorom may prompt for ROM install; this may fail in some environments. If autorom fails, follow Gym/Atari installation instructions.
# Removed '|| true' to see explicit errors from autorom if it fails.
!python -m autorom


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: opencv-python-headless 4.7.0.72
Uninstalling opencv-python-headless-4.7.0.72:
  Successfully uninstalled opencv-python-headless-4.7.0.72
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, which is not installed.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you h

In [ ]:
# Imports and device
import math, random, time, collections, os, sys
from dataclasses import dataclass
from typing import List
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import deque
import cv2
import imageio

# Ensure Atari ROMs are registered
import ale_py.roms

print('torch', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device ->', device)


torch 2.9.0+cu126
device -> cpu


In [ ]:
# ===== USER SETTINGS: edit these before running training =====
ENV_ID = "PongNoFrameskip-v4"   # Atari environment id
SEED = 0
MAX_FRAMES = 2_000_000          # total frames to run (increase for full training)
REPLAY_SIZE = 100_000           # use 1_000_000 for full-scale experiments
BATCH_SIZE = 32
GAMMA = 0.99
LR = 1e-4
REPLAY_START_SIZE = 50_000      # frames before learning starts
TRAIN_FREQ = 4                  # train every N frames
TARGET_SYNC_FREQ = 10_000
FRAME_STACK = 4
EPS_START = 1.0
EPS_FINAL = 0.01
EPS_DECAY_FRAMES = 1_000_000
USE_DUELING = True
USE_DDQN = True
USE_PER = True
USE_NSTEP = True
N_STEPS = 3
SAVE_DIR = "/content/dqn_pong_runs"
os.makedirs(SAVE_DIR, exist_ok=True)
# ============================================================


In [ ]:
# Reproducible seeds
def set_seed(seed:int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
set_seed(SEED)


In [ ]:
# Environment wrappers and preprocessing helpers
import gymnasium as gym

def make_atari_env(env_id=ENV_ID, seed=SEED):
    env = gym.make(env_id, render_mode=None)
    env.reset(seed=seed)
    return env

def preprocess_frame(frame):
    # frame: HxWx3 RGB uint8
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (84,84), interpolation=cv2.INTER_AREA)
    return frame  # uint8

class FrameStack:
    def __init__(self, k):
        self.k = k
        self.deque = deque(maxlen=k)

    def reset(self, obs):
        self.deque.clear()
        for _ in range(self.k):
            self.deque.append(obs)
        return np.stack(self.deque, axis=0)

    def step(self, obs):
        self.deque.append(obs)
        return np.stack(self.deque, axis=0)


In [ ]:
# Experience tuple
@dataclass
class Experience:
    state: np.ndarray
    action: int
    reward: float
    done: bool
    next_state: np.ndarray


In [ ]:
# Replay buffers: uniform and prioritized
class ExperienceBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.pos = 0

    def __len__(self):
        return len(self.buffer)

    def append(self, exp: Experience):
        if len(self.buffer) < self.capacity:
            self.buffer.append(exp)
        else:
            self.buffer[self.pos] = exp
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in idx]
        return batch

class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, alpha: float = 0.6):
        self.capacity = capacity
        self.buffer = []
        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.pos = 0
        self.alpha = alpha

    def __len__(self):
        return len(self.buffer)

    def append(self, exp: Experience, priority: float | None = None):
        if len(self.buffer) < self.capacity:
            self.buffer.append(exp)
            idx = len(self.buffer) - 1
        else:
            idx = self.pos
            self.buffer[self.pos] = exp
        if priority is None:
            p = self.priorities.max() if len(self.buffer) > 0 else 1.0
        else:
            p = priority
        self.priorities[idx] = p
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size: int, beta: float = 0.4):
        prios = self.priorities[:len(self.buffer)].copy()
        probs = prios ** self.alpha
        probs_sum = probs.sum()
        if probs_sum == 0:
            probs = np.ones_like(probs) / len(probs)
        else:
            probs = probs / probs_sum
        indices = np.random.choice(len(self.buffer), batch_size, p=probs, replace=False)
        batch = [self.buffer[i] for i in indices]
        weights = (len(self.buffer) * probs[indices]) ** (-beta)
        weights = weights / (weights.max() + 1e-8)
        return batch, indices, torch.tensor(weights, dtype=torch.float32)

    def update_priorities(self, indices: List[int], new_prios: np.ndarray, eps: float = 1e-5):
        self.priorities[indices] = np.abs(new_prios) + eps


In [ ]:
# Models: DQN and Dueling DQN
class DQN(nn.Module):
    def __init__(self, in_channels=FRAME_STACK, n_actions=6):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
        self.fc1 = nn.Linear(7*7*64, 512)
        self.out = nn.Linear(512, n_actions)

    def forward(self, x):
        x = x.float() / 255.0
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.out(x)

class DuelingDQN(nn.Module):
    def __init__(self, in_channels=FRAME_STACK, n_actions=6):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
            nn.Flatten()
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, 84, 84)
            feat_size = self.conv(dummy).shape[-1]
        self.value = nn.Sequential(nn.Linear(feat_size, 512), nn.ReLU(), nn.Linear(512,1))
        self.adv   = nn.Sequential(nn.Linear(feat_size, 512), nn.ReLU(), nn.Linear(512, n_actions))

    def forward(self, x):
        x = x.float() / 255.0
        f = self.conv(x)
        v = self.value(f)
        a = self.adv(f)
        q = v + (a - a.mean(dim=1, keepdim=True))
        return q


In [ ]:
# Agent classes: Agent and NStepAgent
class Agent:
    def __init__(self, env, buffer, frame_stack_k=FRAME_STACK):
        self.env = env
        self.buffer = buffer
        self.frame_stack_k = frame_stack_k
        self.frame_stack = FrameStack(frame_stack_k)
        self.state = None
        self.total_reward = 0.0

    def reset(self):
        obs, _ = self.env.reset()
        obs = preprocess_frame(obs)
        self.state = self.frame_stack.reset(obs)
        self.total_reward = 0.0
        return self.state

    def _sample_action(self, net, epsilon, device):
        if random.random() < epsilon:
            return int(self.env.action_space.sample())
        else:
            st_v = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                q = net(st_v)
                return int(q.argmax().item())

    def play_step(self, net, device, epsilon=0.0):
        # returns reward if done else None
        if self.state is None:
            state = self.reset()
        action = self._sample_action(net, epsilon, device)
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs = preprocess_frame(obs)
        next_state = self.frame_stack.step(obs)
        done = bool(terminated or truncated)
        self.total_reward += reward
        exp = Experience(state=self.state.copy(), action=action, reward=reward, done=done, next_state=next_state.copy())
        self.buffer.append(exp)
        self.state = next_state
        if done:
            r = self.total_reward
            self.reset()
            return r
        return None

class NStepAgent(Agent):
    def __init__(self, env, buffer, n_steps=3, gamma=GAMMA, frame_stack_k=FRAME_STACK):
        super().__init__(env, buffer, frame_stack_k)
        self.n_steps = n_steps
        self.gamma = gamma
        self.traj = collections.deque()

    def _push_nstep(self):
        R = 0.0
        terminal = False
        for i, (_, _, r, _, d) in enumerate(self.traj):
            R += (self.gamma ** i) * r
            if d:
                terminal = True
                break
        s0, a0, _, _, _ = self.traj[0]
        sN = self.traj[-1][3]
        doneN = self.traj[-1][4] or terminal
        self.buffer.append(Experience(state=s0.copy(), action=a0, reward=float(R), done=bool(doneN), next_state=sN.copy()))

    def play_step(self, net, device, epsilon=0.0):
        if self.state is None:
            self.reset()
        action = self._sample_action(net, epsilon, device)
        obs, reward, terminated, truncated, info = self.env.step(action)
        obs = preprocess_frame(obs)
        next_state = self.frame_stack.step(obs)
        done = bool(terminated or truncated)
        self.total_reward += reward
        self.traj.append((self.state.copy(), action, reward, next_state.copy(), done))
        if len(self.traj) >= self.n_steps:
            self._push_nstep()
            self.traj.popleft()
        self.state = next_state
        if done:
            while self.traj:
                self._push_nstep()
                self.traj.popleft()
            r = self.total_reward
            self.reset()
            return r
        return None


In [ ]:
# Helpers: batch conversion and loss calculation
def batch_to_tensors(batch: List[Experience], device):
    states = np.stack([b.state for b in batch], axis=0)
    next_states = np.stack([b.next_state for b in batch], axis=0)
    actions = np.array([b.action for b in batch], dtype=np.int64)
    rewards = np.array([b.reward for b in batch], dtype=np.float32)
    dones = np.array([b.done for b in batch], dtype=np.bool_)
    states_t = torch.tensor(states, dtype=torch.float32).to(device)
    next_states_t = torch.tensor(next_states, dtype=torch.float32).to(device)
    actions_t = torch.tensor(actions, dtype=torch.int64).to(device)
    rewards_t = torch.tensor(rewards, dtype=torch.float32).to(device)
    dones_t = torch.tensor(dones, dtype=torch.bool).to(device)
    return states_t, actions_t, rewards_t, dones_t, next_states_t

def calc_loss(batch: List[Experience], net: nn.Module, tgt_net: nn.Module, device: torch.device, gamma=GAMMA, use_ddqn=USE_DDQN):
    states_t, actions_t, rewards_t, dones_t, new_states_t = batch_to_tensors(batch, device)
    q_sa = net(states_t).gather(1, actions_t.unsqueeze(-1)).squeeze(-1)
    with torch.no_grad():
        if use_ddqn:
            next_actions = net(new_states_t).argmax(dim=1)
            next_q = tgt_net(new_states_t).gather(1, next_actions.unsqueeze(-1)).squeeze(-1)
        else:
            next_q = tgt_net(new_states_t).max(1)[0]
        next_q[dones_t] = 0.0
        target = rewards_t + gamma * next_q
    return F.mse_loss(q_sa, target)


In [ ]:
# Setup environment, networks, buffer, agent, optimizer
env = make_atari_env(ENV_ID, seed=SEED)
n_actions = env.action_space.n
print('n_actions:', n_actions)
ModelCls = DuelingDQN if USE_DUELING else DQN
net = ModelCls(in_channels=FRAME_STACK, n_actions=n_actions).to(device)
tgt_net = ModelCls(in_channels=FRAME_STACK, n_actions=n_actions).to(device)
tgt_net.load_state_dict(net.state_dict())

if USE_PER:
    buffer = PrioritizedReplayBuffer(REPLAY_SIZE, alpha=0.6)
else:
    buffer = ExperienceBuffer(REPLAY_SIZE)

AgentCls = NStepAgent if USE_NSTEP else Agent
agent = AgentCls(env, buffer, n_steps=N_STEPS, gamma=GAMMA) if USE_NSTEP else AgentCls(env, buffer)
optimizer = optim.Adam(net.parameters(), lr=LR)
print('Setup complete. Model / buffer / agent ready.')


n_actions: 6
Setup complete. Model / buffer / agent ready.


In [ ]:
# Training loop
episode_rewards = []
mean_100_values = []
losses = []
eps_values = []

frame_idx = 0
episode = 0
best_mean_reward = None

state = agent.reset()
start_time = time.time()

while frame_idx < MAX_FRAMES:
    # epsilon linear schedule
    if frame_idx < EPS_DECAY_FRAMES:
        eps = EPS_START + (EPS_FINAL - EPS_START) * (frame_idx / EPS_DECAY_FRAMES)
    else:
        eps = EPS_FINAL
    eps_values.append(eps)

    ret = agent.play_step(net, device, epsilon=eps)
    frame_idx += 1

    # training
    if len(buffer) >= REPLAY_START_SIZE and frame_idx % TRAIN_FREQ == 0:
        if USE_PER:
            batch, indices, is_weights = buffer.sample(BATCH_SIZE, beta=0.4)
            states_t, actions_t, rewards_t, dones_t, new_states_t = batch_to_tensors(batch, device)
            q_sa = net(states_t).gather(1, actions_t.unsqueeze(-1)).squeeze(-1)
            with torch.no_grad():
                if USE_DDQN:
                    next_actions = net(new_states_t).argmax(1)
                    next_q = tgt_net(new_states_t).gather(1, next_actions.unsqueeze(-1)).squeeze(-1)
                else:
                    next_q = tgt_net(new_states_t).max(1)[0]
                next_q[dones_t] = 0.0
                # if using N-step, the rewards in buffer are already N-step aggregated, but we adjust gamma power outside
                gamma_eff = GAMMA ** N_STEPS if USE_NSTEP else GAMMA
                target = rewards_t + gamma_eff * next_q
            td = q_sa - target
            loss_v = (is_weights.to(device) * (td ** 2)).mean()
            optimizer.zero_grad(); loss_v.backward(); nn.utils.clip_grad_norm_(net.parameters(), 10.0); optimizer.step()
            losses.append(float(loss_v.item()))
            buffer.update_priorities(indices, np.abs(td.detach().cpu().numpy()))
        else:
            batch = buffer.sample(BATCH_SIZE)
            gamma_eff = GAMMA ** N_STEPS if USE_NSTEP else GAMMA
            loss_v = calc_loss(batch, net, tgt_net, device, gamma=gamma_eff, use_ddqn=USE_DDQN)
            optimizer.zero_grad(); loss_v.backward(); nn.utils.clip_grad_norm_(net.parameters(), 10.0); optimizer.step()
            losses.append(float(loss_v.item()))

    # target sync
    if frame_idx % TARGET_SYNC_FREQ == 0:
        tgt_net.load_state_dict(net.state_dict())

    # episode done logging
    if ret is not None:
        episode += 1
        episode_rewards.append(ret)
        mean100 = np.mean(episode_rewards[-100:]) if len(episode_rewards) >= 1 else np.mean(episode_rewards)
        mean_100_values.append(mean100)
        if best_mean_reward is None or mean100 > best_mean_reward:
            best_mean_reward = mean100
            torch.save(net.state_dict(), os.path.join(SAVE_DIR, 'best_net.pth'))
        if episode % 10 == 0:
            elapsed = time.time() - start_time
            print(f'Frame: {frame_idx:,}, Episode: {episode}, LastR: {episode_rewards[-1]:.1f}, Mean100: {mean100:.2f}, Eps: {eps:.4f}, Loss(avg100): {np.mean(losses[-100:]) if losses else 0:.4f}, Elapsed: {int(elapsed)}s')


Frame: 32,840, Episode: 10, LastR: -21.0, Mean100: -20.80, Eps: 0.9675, Loss(avg100): 0.0000, Elapsed: 29s
Frame: 65,855, Episode: 20, LastR: -21.0, Mean100: -20.65, Eps: 0.9348, Loss(avg100): 0.0050, Elapsed: 654s


In [ ]:
# Plot learning curves
import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
plt.plot(episode_rewards, alpha=0.25, label='episodic returns')
if mean_100_values:
    plt.plot(mean_100_values, label='mean100')
plt.xlabel('Episodes')
plt.ylabel('Return')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Save final model
torch.save(net.state_dict(), os.path.join(SAVE_DIR, 'final_net.pth'))
print('Saved final model to', os.path.join(SAVE_DIR, 'final_net.pth'))


In [ ]:
# Record short video of agent (use after training)
from gymnasium.wrappers import RecordVideo

def record_video(policy_net, filename, env_id=ENV_ID, max_steps=600, greedy=True):
    vid_dir = os.path.dirname(filename)
    os.makedirs(vid_dir, exist_ok=True)
    env = gym.make(env_id, render_mode='rgb_array')
    obs, _ = env.reset(seed=SEED)
    frame_stack = FrameStack(FRAME_STACK)
    obs_proc = preprocess_frame(obs)
    state = frame_stack.reset(obs_proc)
    images = []
    done = False
    steps = 0
    policy_net = policy_net.to('cpu')
    while not done and steps < max_steps:
        if greedy:
            with torch.no_grad():
                q = policy_net(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
                action = int(q.argmax().item())
        else:
            action = env.action_space.sample()
        obs, rew, terminated, truncated, info = env.step(action)
        img = env.render()
        images.append(img)
        obs_proc = preprocess_frame(obs)
        state = frame_stack.step(obs_proc)
        done = terminated or truncated
        steps += 1
    imageio.mimsave(filename, images, fps=30)
    env.close()
    print('Saved video to', filename)

# Examples (uncomment to run)
record_video(net, '/content/dqn_pong_runs/learned.mp4', greedy=True, max_steps=600)
record_video(net, '/content/dqn_pong_runs/early_random.mp4', greedy=False, max_steps=300)


# Quick README (auto-generated)
This notebook trains a DQN agent on Atari Pong. Toggle variants in the 'USER SETTINGS' cell:
- USE_DUELING: dueling architecture
- USE_DDQN: double DQN target
- USE_PER: prioritized replay
- USE_NSTEP: n-step returns (N_STEPS)

Core files will be saved to `/content/dqn_pong_runs`. Use the video cell to record short clips after training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')